# Updating mutation plots

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings 
import numpy as np
import pandas as pd
import seaborn as sns
import muon as mu


warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, facecolor='white')

In [ ]:
adata = sc.read_h5ad("14_gene_expression.h5ad")

In [ ]:
adata

In [ ]:
adata.obs['timepoint_coarse'].value_counts()

In [ ]:
new_adata_HSC_nohealthy = adata[adata.obs["patient"].str.startswith("61")]
new_adata_HSC_nohealthy.obs["disease_state"].value_counts()

In [ ]:
#check if barcodes are duplicated
new_adata_HSC_nohealthy.obs_names.duplicated().sum()

In [ ]:
sc_mut = pd.read_csv("data/single_cell_mutation_for_priscilla.csv")

In [ ]:
sc_mut = sc_mut.iloc[:,1:]

In [ ]:
sc_mut.set_index("barcode_index", inplace=True)

In [ ]:
sc_mut.fillna("na", inplace=True)
sc_mut

In [ ]:
sc_mut = sc_mut.drop(columns=["BARCODE"])

In [ ]:
missing_ids = sc_mut.index.difference(new_adata_HSC_nohealthy.obs.index)
if not missing_ids.empty:
    print("Warning: The following IDs are in df but not in adata.obs:", missing_ids)

# Proceed with reindexing or merging
sc_mut = sc_mut.reindex(new_adata_HSC_nohealthy.obs.index).fillna("na")
adata_sc_mut = new_adata_HSC_nohealthy.copy()
adata_sc_mut.obs = adata_sc_mut.obs.join(sc_mut)
adata_sc_mut.obs

In [ ]:
adata_sc_mut.obs[['timepoint','timepoint_coarse']].value_counts()

In [ ]:
adata_sc_mut[adata_sc_mut.obs['chr17_76736877_G_A_call'].dropna().index,:].obs[['dataset_name','patient_alias']].value_counts()

In [ ]:
adata_sc_mut

In [ ]:
adata_sc_mut.obs

In [ ]:
print(adata_sc_mut.obs.dtypes)

In [ ]:
adata_sc_mut.obs[adata_sc_mut.obs.select_dtypes('object').columns] = adata_sc_mut.obs[adata_sc_mut.obs.select_dtypes('object').columns].astype(str).astype("category")

In [ ]:
adata_sc_mut.obs

In [ ]:
#adata_sc_mut.write_h5ad("15_adata_sc_mut.h5ad")

# Update zorder dynamically

In [ ]:
adata_sc_mut = sc.read_h5ad("13052025_updated_annotations_colors_scores.h5ad")

In [ ]:
newdf=pd.DataFrame({
    "X_coord_umap":adata_sc_mut.obsm["X_umap"][:,0],
    "Y_coord_umap":adata_sc_mut.obsm["X_umap"][:,1],
    "celltype":adata_sc_mut.obs["celltype_v2"],
    "outcome_C6D28":adata_sc_mut.obs["outcome_C6D28"],
    'outcome_C12D29':adata_sc_mut.obs["outcome_C12D29"],
    "timepoint_coarse":adata_sc_mut.obs["timepoint_coarse"],
    "patient":adata_sc_mut.obs["patient"],
    "patient_alias": adata_sc_mut.obs["patient_alias"],
    'leiden': adata_sc_mut.obs["leiden"],
    'chr17_76736877_G_A_call': adata_sc_mut.obs['chr17_76736877_G_A_call'],
    'chr17_76736877_G_A_mut': adata_sc_mut.obs['chr17_76736877_G_A_mut'],
    'chr17_76736877_G_C_call': adata_sc_mut.obs['chr17_76736877_G_C_call'],
    'chr17_76736877_G_C_mut': adata_sc_mut.obs['chr17_76736877_G_C_mut'],
    'chr17_76736877_G_T_call': adata_sc_mut.obs['chr17_76736877_G_T_call'],
    'chr17_76736877_G_T_mut': adata_sc_mut.obs['chr17_76736877_G_T_mut'],
    'chr17_7674250_C_T_call': adata_sc_mut.obs['chr17_7674250_C_T_call'],
    'chr17_7674250_C_T_mut': adata_sc_mut.obs['chr17_7674250_C_T_mut'],
    'chr17_7675082_G_T_call': adata_sc_mut.obs['chr17_7675082_G_T_call'],
    'chr17_7675082_G_T_mut': adata_sc_mut.obs['chr17_7675082_G_T_mut'],
    'chr17_7676051_G_C_call': adata_sc_mut.obs['chr17_7676051_G_C_call'],
    'chr17_7676051_G_C_mut': adata_sc_mut.obs['chr17_7676051_G_C_mut']
})
newdf

In [ ]:
sc.pl.umap(adata_sc_mut, color=['chr17_76736877_G_A_call'])

In [ ]:
from MDS_figure2_dicts import *
newdf['ctgrey'] = "#c8c8c8"

{'chr17:76736877_G>C': 'SRSF2',
 'chr17:7675082_G>T': 'TP53',
 'chr17:76736877_G>A': 'SRSF2',
 'chr17:76736877_G>T': 'SRSF2',
 'chr17:7674250_C>T': 'TP53',
 'chr17:7676051_G>C': 'TP53'}

In [ ]:
newdf[newdf['patient_alias'] == "P03"]

In [ ]:
newdf[(newdf['patient_alias'] == "P03") & (newdf['timepoint_coarse'] == 'mid')]['outcome_C6D28'].value_counts().idxmax()

In [ ]:
newdf['timepoint_coarse'].value_counts()

In [ ]:
newdf.columns[newdf.columns.str.contains('call')]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import pandas as pd
import re

# Define mappings
size_mapping = {'unknown': 10, 'wt': 15, 'mt': 15}
color_map = {'unknown': '#fe9003', 'wt': '#2cace3', 'mt': '#e90c8b'}
zorder_map = {'unknown': 1, 'wt': 2, 'mt': 3}

for chr in newdf.columns[newdf.columns.str.contains('call')]:
    for p in newdf['patient_alias'].unique():
        valid_axes_data = []  # Track valid subplots

        for timepoint in ['pre', 'mid', 'post', 'progression']:
            df = newdf[newdf['patient_alias'] == p]
            data = df[df['timepoint_coarse'] == timepoint]

            # Count mutations
            counts = data[chr].value_counts()
            unknown_count = counts.get("unknown", 0)
            wt_count = counts.get("wt", 0)
            mt_count = counts.get("mt", 0)

            # Always fix 'unknown' as zorder = 0
            dynamic_zorder_map = {'unknown': 1}

            # Get counts for mt and wt
            mt_count = counts.get('mt', 0)
            wt_count = counts.get('wt', 0)

            # Sort 'mt' and 'wt' by descending count
            sorted_mutations = sorted(
                [('mt', mt_count), ('wt', wt_count)],
                key=lambda x: x[1], reverse=True)

            # Assign zorder starting from 1 (above unknown)
            for i, (mut_type, _) in enumerate(sorted_mutations, start=2):
                dynamic_zorder_map[mut_type] = i

            # Skip subplot if all counts are zero
            if wt_count == 0 and unknown_count == 0 and mt_count == 0:
                continue

            valid_axes_data.append((timepoint, data))

        # If no valid subplots exist, skip this figure
        if not valid_axes_data:
            continue

        # Dynamically create figure with the correct number of subplots
        fig, axes = plt.subplots(1, len(valid_axes_data), figsize=(6 * len(valid_axes_data), 6), dpi=300)

        # Ensure axes is iterable (if only one subplot, `axes` is not a list)
        if len(valid_axes_data) == 1:
            axes = [axes]

        for ax, (timepoint, data) in zip(axes, valid_axes_data):
            # Plot background UMAP scatter (rasterized for efficiency)
            ax.scatter(newdf['X_coord_umap'], newdf['Y_coord_umap'], 
                       c=newdf['ctgrey'], s=5, alpha=0.3, rasterized=True)

            # Overlay mutation points (rasterized for efficiency)
            for category in ['unknown', 'wt', 'mt']:
                subset = data[data[chr] == category]
                if not subset.empty:
                    ax.scatter(subset['X_coord_umap'], subset['Y_coord_umap'],
                               s=size_mapping[category], c=color_map[category],
                               zorder=dynamic_zorder_map[category], edgecolors='none', 
                               label=category, rasterized=True)

            # Display counts
            counts = data[chr].value_counts()
            text_str = f"unknown: {counts.get('unknown', 0)}\nwt: {counts.get('wt', 0)}\nmt: {counts.get('mt', 0)}"
            ax.text(0.95, 0.95, text_str, transform=ax.transAxes, ha="right", va="top", fontsize=15)

            ax.set_title(f"{p} {timepoint}", fontsize=25)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.grid(False)

        # Adjust layout dynamically
        fig.suptitle(chr, fontsize=40)
        fig.tight_layout()

        # Create legend
        handles = [mlines.Line2D([0], [0], marker='o', linestyle='', markersize=20,
                                markerfacecolor=color_map[label],markeredgecolor=color_map[label],
                                label=label) for label in color_map.keys()]
        fig.legend(handles=handles, title='Mutations', loc="upper center",
                   bbox_to_anchor=(0.5, -0.1), fontsize=20, ncol=3)

        # Save figure as PDF with rasterization
        savefigname = re.sub(r"[^\w]", "_", str(chr))
        fig.savefig(f"figures/mutation_calls/{savefigname}_{p}_UMAP_v3.pdf", 
                    format='pdf', bbox_inches='tight', dpi=200)
        #fig.savefig(f"figures/mutation_calls/{savefigname}_{p}_UMAP_v3.png", 
        #            format='png', bbox_inches='tight', dpi=300)            

        plt.close(fig)


In [ ]:
#map mutations across the proteome

mdata_sc_mut = mu.read_h5mu("06052025_gex_adt_celltypev2_updated_colors.h5mu")

In [ ]:
mdata_sc_mut['protein'].obs[""]

In [ ]:
sc.pl.umap(adata_sc_mut[(adata_sc_mut.obs['celltype_v2'] == "HSC") & (adata_sc_mut.obs['patient_alias'] != "P09")], color=["phase","patient_alias"], title="cell cycle phase in HSC in everyone else")

In [ ]:
hsc_p09 = adata_sc_mut[(adata_sc_mut.obs['celltype_v2'] == "HSC") & (adata_sc_mut.obs['patient_alias'] == "P09")]
hsc_else = adata_sc_mut[(adata_sc_mut.obs['celltype_v2'] == "HSC") & (adata_sc_mut.obs['patient_alias'] != "P09")]

In [ ]:
hsc_p09.obs["phase"].value_counts()

In [ ]:
90/2331

In [ ]:
hsc_else.obs["phase"].value_counts()

In [ ]:
13/467

# Updating celltype to include HSC_P09 and additional mutation calls

In [ ]:
mdata = mu.read_h5mu("06052025_gex_adt_celltypev2_updated_colors.h5mu")
mdata

In [ ]:
mdata.mod['rna'].obs['celltype_v3'] = adata.obs['celltype_v2'].copy()
adata.obs['celltype_v3'] = adata.obs['celltype_v3'].cat.add_categories(['HSC_P09'])
mdata.mod['rna'].obs.loc[
    (mdata.mod['rna'].obs['patient_alias'] == 'P09') & (adata.obs['celltype_v3'] == 'HSC'),
    'celltype_v3'
] = 'HSC_P09'
mdata.mod['rna'].obs[['patient_alias', 'celltype_v3']]

In [ ]:
mdata.update()

In [ ]:
mdata.mod['protein'].obs['celltype_v3'] = mdata.mod['protein'].obs['celltype_v2'].copy()
mdata.update()

In [ ]:
mutation_file2 = pd.read_csv("data/updated_mutations_for_priscilla_17052025.csv")
mutation_file2

In [ ]:
mutation_file2['temp_index'] = mutation_file2['dataset_name']+"_"+mutation_file2['Unnamed: 0']
mutation_file2.set_index('temp_index', inplace=True)
mutation_file2

In [ ]:
#check that both indexes are the same
mutation_file2.index.equals(mdata.mod['rna'].obs.index)

In [ ]:
mdata.mod['rna'].obs = pd.concat([mdata.mod['rna'].obs, mutation_file2.iloc[:,-8:]], axis=1)

In [ ]:
mdata

In [ ]:
mdata['rna'].obs['chr21_34859485_C_T_mt_updated'].value_counts()

In [ ]:
#mdata.write("17052025_celltypev3_mutations.h5mu")

In [ ]:
#mdata['rna'].write_h5ad("17052025_celltypev3_mutations.h5ad")

# Update the mutation UMAPS

In [ ]:
adata = sc.read_h5ad("17052025_celltypev3_mutations.h5ad")

In [ ]:
adata

In [ ]:
# get list of column names containing call, then make list into a dictionary

call_list = adata.obs.columns[adata.obs.columns.str.contains("call|_mt_", regex=True)].tolist()
call_list

In [ ]:
adata.obs['chr1_114716126_C_T_mt_updated'].value_counts()

In [ ]:
mut_df=pd.DataFrame({
    "X_coord_umap":adata.obsm["X_umap"][:,0],
    "Y_coord_umap":adata.obsm["X_umap"][:,1],
    "celltype":adata.obs["celltype_v3"],
    "outcome_C6D28":adata.obs["outcome_C6D28"],
    'outcome_C12D29':adata.obs["outcome_C12D29"],
    "timepoint_coarse":adata.obs["timepoint_coarse"],
    "patient":adata.obs["patient"],
    "patient_alias": adata.obs["patient_alias"],
    'leiden': adata.obs["leiden"],
    'chr17_76736877_G_A_call': adata.obs['chr17_76736877_G_A_call'],
    'chr17_76736877_G_C_call': adata.obs['chr17_76736877_G_C_call'],
    'chr17_76736877_G_T_call': adata.obs['chr17_76736877_G_T_call'],
    'chr17_7674250_C_T_call': adata.obs['chr17_7674250_C_T_call'],
    'chr17_7675082_G_T_call': adata.obs['chr17_7675082_G_T_call'],
    'chr17_7676051_G_C_call': adata.obs['chr17_7676051_G_C_call'],
    'chr17_76736877_G_T_call_updated': adata.obs['chr17_76736877_G_T_call_updated'],
    'chr17_76736877_G_A_call_updated': adata.obs['chr17_76736877_G_A_call_updated'],
    'chr17_76736877_G_C_call_updated': adata.obs['chr17_76736877_G_C_call_updated'],
    'chr17_7674250_C_T_call_updated': adata.obs['chr17_7674250_C_T_call_updated'],
    'chr17_7675082_G_T_call_updated': adata.obs['chr17_7675082_G_T_call_updated'],
    'chr21_34886879_G_C_mt_updated':adata.obs['chr21_34886879_G_C_mt_updated'],
    'chr1_114716126_C_T_mt_updated':adata.obs['chr1_114716126_C_T_mt_updated'],
    'chr21_34859485_C_T_mt_updated':adata.obs['chr21_34859485_C_T_mt_updated'],
})
mut_df

In [ ]:
#check that everything is in order
sc.pl.umap(adata, color=['chr21_34859485_C_T_mt_updated'])

In [ ]:
from MDS_figure2_dicts import *
mut_df['ctgrey'] = "#e2e2e2"

In [ ]:
mut_df

In [ ]:
mut_df.columns[mut_df.columns.str.contains('updated')]

In [ ]:
adata.obs[['patient_alias','chr21_34886879_G_C_mt_updated']].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import pandas as pd
import re

# Define mappings
size_mapping = {'unknown': 10, 'wt': 15, 'mt': 15}
color_map = {'unknown': "#808080", 'wt': "#4997c9", 'mt': "#c70e77"}
zorder_map = {'unknown': 1, 'wt': 2, 'mt': 3}


for chr in mut_df.columns[mut_df.columns.str.contains('updated')]:
    print(chr)
    for p in mut_df['patient_alias'].unique():
        print(p)
        valid_axes_data = []  # Track valid subplots

        for timepoint in ['pre', 'mid', 'post', 'progression']:
            df = mut_df[mut_df['patient_alias'] == p]
            data = df[df['timepoint_coarse'] == timepoint]

            # Count mutations
            counts = data[chr].value_counts()
            unknown_count = counts.get("unknown", 0)
            wt_count = counts.get("wt", 0)
            mt_count = counts.get("mt", 0)

            # Always fix 'unknown' as zorder = 0
            dynamic_zorder_map = {'unknown': 1}

            # Get counts for mt and wt
            mt_count = counts.get('mt', 0)
            wt_count = counts.get('wt', 0)

            # Sort 'mt' and 'wt' by descending count
            sorted_mutations = sorted(
                [('mt', mt_count), ('wt', wt_count)],
                key=lambda x: x[1], reverse=True)

            # Assign zorder starting from 1 (above unknown)
            for i, (mut_type, _) in enumerate(sorted_mutations, start=2):
                dynamic_zorder_map[mut_type] = i

            # Skip subplot if all counts are zero
            if wt_count == 0 and unknown_count == 0 and mt_count == 0:
                continue

            valid_axes_data.append((timepoint, data))

        # If no valid subplots exist, skip this figure
        if not valid_axes_data:
            continue

        # Dynamically create figure with the correct number of subplots
        fig, axes = plt.subplots(1, len(valid_axes_data), figsize=(6 * len(valid_axes_data), 6), dpi=300)

        # Ensure axes is iterable (if only one subplot, `axes` is not a list)
        if len(valid_axes_data) == 1:
            axes = [axes]

        for ax, (timepoint, data) in zip(axes, valid_axes_data):
            # Plot background UMAP scatter (rasterized for efficiency)
            ax.scatter(mut_df['X_coord_umap'], mut_df['Y_coord_umap'], 
                       c=mut_df['ctgrey'], s=5, alpha=0.3, rasterized=True)

            # Overlay mutation points (rasterized for efficiency)
            for category in ['unknown', 'wt', 'mt']:
                subset = data[data[chr] == category]
                if not subset.empty:
                    ax.scatter(subset['X_coord_umap'], subset['Y_coord_umap'],
                               s=size_mapping[category], c=color_map[category],
                               zorder=dynamic_zorder_map[category], edgecolors='none', 
                               label=category, rasterized=True)

            # Display counts
            counts = data[chr].value_counts()
            text_str = f"unknown: {counts.get('unknown', 0)}\nwt: {counts.get('wt', 0)}\nmt: {counts.get('mt', 0)}"
            ax.text(0.95, 0.95, text_str, transform=ax.transAxes, ha="right", va="top", fontsize=15)

            ax.set_title(f"{p} {timepoint}", fontsize=25)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.grid(False)

        # Adjust layout dynamically
        fig.suptitle(chr, fontsize=40)
        fig.tight_layout()

        # Create legend
        handles = [mlines.Line2D([0], [0], marker='o', linestyle='', markersize=20,
                                markerfacecolor=color_map[label],markeredgecolor=color_map[label],
                                label=label) for label in color_map.keys()]
        fig.legend(handles=handles, title='Mutations', loc="upper center",
                   bbox_to_anchor=(0.5, -0.1), fontsize=20, ncol=3)

        # Save figure as PDF with rasterization
        savefigname = re.sub(r"[^\w]", "_", str(chr))
        fig.savefig(f"figures/mutation_calls_v3/{savefigname}_{p}_UMAP.pdf", 
                    format='pdf', bbox_inches='tight', dpi=200)
        #fig.savefig(f"figures/mutation_calls/{savefigname}_{p}_UMAP_v3.png", 
        #            format='png', bbox_inches='tight', dpi=300)            

        plt.close(fig)


# Update some calls from unknown to WT

In [ ]:
adata = sc.read_h5ad("17052025_celltypev3_mutations.h5ad")
adata

In [ ]:
mt_file = pd.read_csv("data/tp53_incorrect_genotyping.csv")
mt_file

In [ ]:
adata

In [ ]:
mt_file['mut_column'] = mt_file['TP53_mutation']+'_call_updated'

In [ ]:
mt_file

In [ ]:
for i,v in enumerate(mt_file['BARCODE']):
    pos = mt_file.iloc[i]
    mask = ((adata.obs['BARCODE'] == pos['BARCODE']) & 
            (adata.obs['timepoint'] == pos['timepoint']) & 
            (adata.obs['celltype_v2'] == pos['celltype_v2']) & 
            (adata.obs['patient_alias'] == pos['patient_alias']))
    adata.obs.loc[mask, pos['mut_column']] = "wt"
    print(adata.obs.loc[mask, pos['mut_column']])
    

In [ ]:
adata.obs.loc[mask]

In [ ]:
adata.write_h5ad("11072025_celltypev3_updated_mutations.h5ad")

In [ ]:
mut_df=pd.DataFrame({
    "X_coord_umap":adata.obsm["X_umap"][:,0],
    "Y_coord_umap":adata.obsm["X_umap"][:,1],
    "celltype":adata.obs["celltype_v3"],
    "outcome_C6D28":adata.obs["outcome_C6D28"],
    'outcome_C12D29':adata.obs["outcome_C12D29"],
    "timepoint_coarse":adata.obs["timepoint_coarse"],
    "patient":adata.obs["patient"],
    "patient_alias": adata.obs["patient_alias"],
    'leiden': adata.obs["leiden"],
    'chr17_76736877_G_A_call': adata.obs['chr17_76736877_G_A_call'],
    'chr17_76736877_G_C_call': adata.obs['chr17_76736877_G_C_call'],
    'chr17_76736877_G_T_call': adata.obs['chr17_76736877_G_T_call'],
    'chr17_7674250_C_T_call': adata.obs['chr17_7674250_C_T_call'],
    'chr17_7675082_G_T_call': adata.obs['chr17_7675082_G_T_call'],
    'chr17_7676051_G_C_call': adata.obs['chr17_7676051_G_C_call'],
    'chr17_76736877_G_T_call_updated': adata.obs['chr17_76736877_G_T_call_updated'],
    'chr17_76736877_G_A_call_updated': adata.obs['chr17_76736877_G_A_call_updated'],
    'chr17_76736877_G_C_call_updated': adata.obs['chr17_76736877_G_C_call_updated'],
    'chr17_7674250_C_T_call_updated': adata.obs['chr17_7674250_C_T_call_updated'],
    'chr17_7675082_G_T_call_updated': adata.obs['chr17_7675082_G_T_call_updated'],
    'chr21_34886879_G_C_mt_updated':adata.obs['chr21_34886879_G_C_mt_updated'],
    'chr1_114716126_C_T_mt_updated':adata.obs['chr1_114716126_C_T_mt_updated'],
    'chr21_34859485_C_T_mt_updated':adata.obs['chr21_34859485_C_T_mt_updated'],
})
mut_df

In [ ]:
from MDS_figure2_dicts import *
mut_df['ctgrey'] = "#e2e2e2"

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import pandas as pd
import re

# Define mappings
size_mapping = {'unknown': 10, 'wt': 15, 'mt': 15}
color_map = {'unknown': "#808080", 'wt': "#4997c9", 'mt': "#c70e77"}
zorder_map = {'unknown': 1, 'wt': 2, 'mt': 3}

tp53 = ['chr17_7674250_C_T_call_updated',
        'chr17_7675082_G_T_call_updated']
for chr in tp53:
    print(chr)
    for p in ['P03','P18']:
        print(p)
        valid_axes_data = []  # Track valid subplots

        for timepoint in ['pre', 'mid', 'post', 'progression']:
            df = mut_df[mut_df['patient_alias'] == p]
            data = df[df['timepoint_coarse'] == timepoint]

            # Count mutations
            counts = data[chr].value_counts()
            unknown_count = counts.get("unknown", 0)
            wt_count = counts.get("wt", 0)
            mt_count = counts.get("mt", 0)

            # Always fix 'unknown' as zorder = 0
            dynamic_zorder_map = {'unknown': 1}

            # Get counts for mt and wt
            mt_count = counts.get('mt', 0)
            wt_count = counts.get('wt', 0)

            # Sort 'mt' and 'wt' by descending count
            sorted_mutations = sorted(
                [('mt', mt_count), ('wt', wt_count)],
                key=lambda x: x[1], reverse=True)

            # Assign zorder starting from 1 (above unknown)
            for i, (mut_type, _) in enumerate(sorted_mutations, start=2):
                dynamic_zorder_map[mut_type] = i

            # Skip subplot if all counts are zero
            if wt_count == 0 and unknown_count == 0 and mt_count == 0:
                continue

            valid_axes_data.append((timepoint, data))

        # If no valid subplots exist, skip this figure
        if not valid_axes_data:
            continue

        # Dynamically create figure with the correct number of subplots
        fig, axes = plt.subplots(1, len(valid_axes_data), figsize=(6 * len(valid_axes_data), 6), dpi=300)

        # Ensure axes is iterable (if only one subplot, `axes` is not a list)
        if len(valid_axes_data) == 1:
            axes = [axes]

        for ax, (timepoint, data) in zip(axes, valid_axes_data):
            # Plot background UMAP scatter (rasterized for efficiency)
            ax.scatter(mut_df['X_coord_umap'], mut_df['Y_coord_umap'], 
                       c=mut_df['ctgrey'], s=5, alpha=0.3, rasterized=True)

            # Overlay mutation points (rasterized for efficiency)
            for category in ['unknown', 'wt', 'mt']:
                subset = data[data[chr] == category]
                if not subset.empty:
                    ax.scatter(subset['X_coord_umap'], subset['Y_coord_umap'],
                               s=size_mapping[category], c=color_map[category],
                               zorder=dynamic_zorder_map[category], edgecolors='none', 
                               label=category, rasterized=True)

            # Display counts
            counts = data[chr].value_counts()
            text_str = f"unknown: {counts.get('unknown', 0)}\nwt: {counts.get('wt', 0)}\nmt: {counts.get('mt', 0)}"
            ax.text(0.95, 0.95, text_str, transform=ax.transAxes, ha="right", va="top", fontsize=15)

            ax.set_title(f"{p} {timepoint}", fontsize=25)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.grid(False)

        # Adjust layout dynamically
        fig.suptitle(chr, fontsize=40)
        fig.tight_layout()

        # Create legend
        handles = [mlines.Line2D([0], [0], marker='o', linestyle='', markersize=20,
                                markerfacecolor=color_map[label],markeredgecolor=color_map[label],
                                label=label) for label in color_map.keys()]
        fig.legend(handles=handles, title='Mutations', loc="upper center",
                   bbox_to_anchor=(0.5, -0.1), fontsize=20, ncol=3)

        # Save figure as PDF with rasterization
        savefigname = re.sub(r"[^\w]", "_", str(chr))
        fig.savefig(f"figures/mutation_calls_v4/{savefigname}_{p}_UMAP.pdf", 
                    format='pdf', bbox_inches='tight', dpi=200)
        #fig.savefig(f"figures/mutation_calls/{savefigname}_{p}_UMAP_v3.png", 
        #            format='png', bbox_inches='tight', dpi=300)            

        plt.close(fig)
